# Module 9: Designing a Reproducible Data Integration Pipeline

**Unit E · Week 9 (part 1)** · Track 1 — Data Integration, Standards, Metadata & Quality

Assembles ingest, standardize, validate, document, and publish into one reproducible, re-runnable pipeline — and this is the one that matters most for this course: its published output is data/processed/track2_dataset.csv, the exact file Track 2 reads.

## Learning objectives

- Assemble ingest, standardize, validate, document, and publish steps into one reproducible, re-runnable pipeline instead of manual point-and-click steps.
- Apply basic version control to pipeline outputs via dated, versioned filenames and a run log.


## Setup

This notebook reads the raw practice files in `../../../data/raw/`, built by
`data/make_track1_sources.py` — three partner-style exports shaped like a
real regional hub's source-system landscape (an ERP/financial export, a
survey-platform export, and an HDX-style pull), plus the P-code gazetteer
used to reconcile them. All four are **synthetic**; see `data/README.md`.
Run `python3 data/make_track1_sources.py` once from the repo root before
working through this notebook if those files aren't there yet.

## Lesson content

- Why "click-ops" data preparation breaks down at scale or on handover: a chain of manual spreadsheet edits is not auditable and does not survive staff turnover; encoding every step in a script makes it re-runnable and reviewable.
- Mapping pipeline stages to the modules already completed: **ingest** (Module 2) -> **standardize** (Module 4) -> **validate** (Module 8) -> **document** (Module 6) -> **publish** a versioned output.
- Lightweight orchestration appropriate to a regional hub's scale: a single scheduled script with a run log is usually enough.
- Basic data versioning: dated output filenames so a bad refresh can be identified and rolled back rather than silently overwriting the last good version.

In [1]:
import datetime as dt
import pandas as pd
import pandera.pandas as pa
from pandera.pandas import Column, Check

run_log = []

def step(name, fn):
    try:
        result = fn()
        run_log.append({"step": name, "status": "OK"})
        return result
    except Exception as e:
        run_log.append({"step": name, "status": f"FAILED: {e}"})
        raise

ALIASES = {"Nyarugenge Dist.": "Nyarugenge", "Rwamagana ": "Rwamagana"}

def ingest():
    return (
        pd.read_csv("../../../data/raw/partner_a_finance_export.csv"),
        pd.read_csv("../../../data/raw/partner_b_survey_export.csv"),
        pd.read_csv("../../../data/raw/partner_c_hdx_pull.csv"),
        pd.read_csv("../../../data/raw/cod_ab_gazetteer.csv"),
    )

def standardize(sources):
    a, b, c, gaz = sources
    a["reg"] = a["reg"].str.strip().replace(ALIASES)
    b["region_name"] = b["region_name"].str.strip().replace(ALIASES)
    a["date"] = pd.to_datetime(a["date"], format="%d/%m/%Y")
    b["date"] = pd.to_datetime(b["collection_date"], format="%d-%b-%y")
    c["date"] = pd.to_datetime(c["date"])

    a = a.merge(gaz, left_on="reg", right_on="district_name", how="left")
    b = b.merge(gaz, left_on="region_name", right_on="district_name", how="left")

    merged = (
        c.merge(a[["district_pcode", "date", "rev"]], on=["district_pcode", "date"], how="left")
         .merge(b[["district_pcode", "date", "feature_1", "feature_2"]], on=["district_pcode", "date"], how="left")
         .merge(gaz[["district_pcode", "district_name", "province"]], on="district_pcode", how="left")
    )
    published = merged.rename(columns={"rev": "value", "district_name": "district"})[
        ["date", "province", "district", "district_pcode", "indicator", "value", "feature_1", "feature_2", "outcome"]
    ]
    published["date"] = published["date"].dt.strftime("%Y-%m-%d")
    for col in ("value", "feature_1", "feature_2"):
        published[col] = published[col].round(2)
    return published.sort_values(["district", "date"]).reset_index(drop=True)

def validate(df):
    gazetteer_codes = set(pd.read_csv("../../../data/raw/cod_ab_gazetteer.csv")["district_pcode"])
    schema = pa.DataFrameSchema({
        "district_pcode": Column(str, Check.isin(gazetteer_codes), nullable=False),
        "value": Column(float, Check.ge(0)),
        "date": Column(str, Check.str_matches(r"^\d{4}-\d{2}-\d{2}$")),
    })
    schema.validate(df, lazy=True)

def document(df):
    dictionary = pd.DataFrame({
        "column": df.columns, "dtype": [str(t) for t in df.dtypes],
        "pct_missing": [round(df[c].isna().mean() * 100, 1) for c in df.columns],
    })
    dictionary.to_csv("../../../data/processed/track2_dataset_DICTIONARY.csv", index=False)

sources = step("ingest", ingest)
published = step("standardize", lambda: standardize(sources))
step("validate", lambda: validate(published))
step("document", lambda: document(published))

def publish(df):
    # Two outputs: the exact file Track 2 reads, and a dated, versioned
    # copy for audit history — the "publish" and "version" halves of
    # this module's lesson content.
    df.to_csv("../../../data/processed/track2_dataset.csv", index=False)
    timestamp = dt.date.today().isoformat()
    df.to_csv(f"../../../data/processed/published_regional_data_{timestamp}.csv", index=False)

step("publish", lambda: publish(published))
pd.DataFrame(run_log).to_csv("../../../data/processed/run_log.csv", index=False)
print(pd.DataFrame(run_log))
print(f"\nPublished {len(published):,} rows to data/processed/track2_dataset.csv")
print("This is the exact file Track 2 Module 1 reads — the handoff, made real.")

          step status
0       ingest     OK
1  standardize     OK
2     validate     OK
3     document     OK
4      publish     OK

Published 240 rows to data/processed/track2_dataset.csv
This is the exact file Track 2 Module 1 reads — the handoff, made real.


### Verifying the handoff

If Track 2's `data/make_sample_data.py` has already been run in this repo, `data/processed/track2_dataset.csv` should now contain the same values either way — this pipeline and that shortcut script are seeded identically. Run the cell below to confirm.

In [2]:
import pandas as pd

check = pd.read_csv("../../../data/processed/track2_dataset.csv")
print(f"{len(check):,} rows · {check['district'].nunique()} districts · "
      f"{check['date'].min()} to {check['date'].max()}")
check.head()

240 rows · 10 districts · 2023-01-01 to 2024-12-01


,date,province,district,district_pcode,indicator,value,feature_1,feature_2,outcome
0,2023-01-01,Kigali City,Gasabo,SIM-GAS,sample_wellbeing_index,39.37,82.01,19.22,1
1,2023-02-01,Kigali City,Gasabo,SIM-GAS,sample_wellbeing_index,45.24,49.99,17.62,1
2,2023-03-01,Kigali City,Gasabo,SIM-GAS,sample_wellbeing_index,41.16,46.66,19.93,0
3,2023-04-01,Kigali City,Gasabo,SIM-GAS,sample_wellbeing_index,42.91,49.17,20.35,0
4,2023-05-01,Kigali City,Gasabo,SIM-GAS,sample_wellbeing_index,42.05,51.04,20.35,0


## Your turn

Run this pipeline end-to-end on a fresh clone (after `python3 data/make_track1_sources.py`), confirm the run log shows every stage OK, and diff the published `track2_dataset.csv` against the copy produced by `data/make_sample_data.py` — they should match exactly.

**Formative assessment.** Pipeline runs successfully end-to-end on a held-out test dataset (graded live); run log correctly captures pass/fail of each stage.